In [6]:
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm import tqdm

In [7]:
dataset = pd.read_parquet('train-00000-of-00001-1ae224438dce829b.parquet')
dataset
results_df = pd.DataFrame(columns=["instruction", "reference", "generated", "bleu_score"])

In [8]:
def load_model_and_tokenizer(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(model_path)
    return model, tokenizer

In [9]:
def generate_response(model, tokenizer, prompt, max_length=256):
    # Tokenize the input
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Initialize the progress bar
    total_steps = max_length  # Number of decoding steps corresponds to max_length
    with tqdm(total=total_steps, desc="Generating response", unit="step") as pbar:
        # Generate the response
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=5,
            early_stopping=True,
            # Update the progress bar on each decoding step
            output_scores=True, 
            return_dict_in_generate=True,
        )
        
        # Update the progress bar
        for _ in range(len(outputs.sequences[0])):
            pbar.update(1)
    
    # Decode the generated response
    response = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    return response

In [ ]:
def evaluate_model(dataset, model, tokenizer):
    global results_df
    bleu_scores = []
    for idx in range(len(dataset)):
        instruction = dataset.at[idx, "instruction"]
        reference = dataset.at[idx, "output"] 
        generated_text = generate_response(model, tokenizer, instruction)

        bleu_score = sentence_bleu(reference, generated_text, weights=(0.25,0.25,0.25,0.25))
        bleu_scores.append(bleu_score)

        print(f"Input: {instruction}")
        print(f"Reference: {reference}")
        print(f"Generated: {generated_text}")
        print(f"BLEU Score: {bleu_score}\n")
                # DataFrame에 결과 추가
        results_df = pd.concat([results_df, pd.DataFrame({
            "instruction": [instruction],
            "reference": [reference],
            "generated": [generated_text],
            "bleu_score": [bleu_score]
        })], ignore_index=True)

    # CSV 파일로 저장
    results_df.to_csv("EEVE-Korean-Instruct-10.8B-v1.0_evaluation.csv", index=False)

In [ ]:
if __name__ == "__main__":
    parquet_path = "train-00000-of-00001-1ae224438dce829b.parquet"  
    model_path = "yanolja/EEVE-Korean-Instruct-10.8B-v1.0"  

    model = AutoModelForCausalLM.from_pretrained("yanolja/EEVE-Korean-Instruct-10.8B-v1.0")
    tokenizer = AutoTokenizer.from_pretrained("yanolja/EEVE-Korean-Instruct-10.8B-v1.0")

    # 평가 실행
    evaluate_model(dataset, model, tokenizer)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Could not cache non-existence of file. Will ignore error and continue. Error: [Errno 13] Permission denied: '/home/jj/.cache/huggingface/hub/models--yanolja--EEVE-Korean-Instruct-10.8B-v1.0/.no_exist/f10d58e8bb7fd949948a3db2d7bd42ff65cc0999/generation_config.json'
Generating response:  79%|███████▉  | 202/256 [1:10:07<18:44, 20.83s/step]    
/home/jj/.local/lib/python3.9/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/jj/.local/lib/python3.9/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()


Input: 보드 게임 스피너는 $A$, $B$, $C$로 표시된 세 부분으로 나뉩니다. 스피너가 $A$에 떨어질 확률은 $\frac{1}{3}$이고, 스피너가 $B$에 떨어질 확률은 $\frac{5}{12}$입니다.  스피너가 $C$에 착륙할 확률은 얼마입니까? 답을 공통 분수로 표현하세요.
Reference: 모든 가능한 결과의 확률의 합이 1$이므로, 스피너가 $C$에 착륙할 확률을 구하려면 스피너가 $A$와 $B$에 착륙할 확률을 1$에서 빼야 합니다. 이를 방정식으로 쓸 수 있습니다: $P(C) = 1 - P(A) - P(B)$. P(A) = \frac{1}{3}$, $P(B) = \frac{5}{12}$라는 것을 알고 있으므로 이 값을 방정식에 대입하여 단순화할 수 있습니다. 결과는 다음과 같습니다: P(C) = 1 - \frac{1}{3} - frac{5}{12} = \frac{12}{12} - frac{4}{12} - frac{5}{12} = \frac{3}{12}$. 분자와 분모를 $3$로 나누면 이 분수를 줄일 수 있습니다: P(C) = \frac{1}{4}$입니다.
Generated: 보드 게임 스피너는 $A$, $B$, $C$로 표시된 세 부분으로 나뉩니다. 스피너가 $A$에 떨어질 확률은 $\frac{1}{3}$이고, 스피너가 $B$에 떨어질 확률은 $\frac{5}{12}$입니다.  스피너가 $C$에 착륙할 확률은 얼마입니까? 답을 공통 분수로 표현하세요.
보드 게임 스피너는 $A$, $B$, $C$로 표시된 세 부분으로 나뉩니다. 스피너가 $A$에 떨어질 확률은 $\frac{1}{3}$이고, 스피너가 $B$에 떨어질 확률은 $\frac{5}{12}$입니다. 스피너가 $C$에 착륙할 확률은 얼마입니까? 답을 공통 분수로 표현하세요.
BLEU Score: 1.0951456330295974e-231



Generating response: 100%|██████████| 256/256 [1:18:49<00:00, 18.47s/step]    


Input: 저희 학교 수학 클럽에는 남학생 6명과 여학생 8명이 있습니다.  주 수학 경시대회에 파견할 팀을 선발해야 합니다. 팀에 6명이 필요합니다.  제한 없이 팀을 몇 가지 방법으로 선발할 수 있나요?
Reference: 14명 중 6명을 선택해야 하는데 순서는 중요하지 않습니다. 이것은 순열 문제가 아니라 조합 문제입니다. 조합의 공식은 nCr = n! / (r! * (n-r)!)이며, 여기서 n은 총 선택의 개수이고 r은 선택의 개수입니다. 숫자를 연결하면 14C6 = 14! / (6! * 8!) = 3003.
Generated: 저희 학교 수학 클럽에는 남학생 6명과 여학생 8명이 있습니다.  주 수학 경시대회에 파견할 팀을 선발해야 합니다. 팀에 6명이 필요합니다.  제한 없이 팀을 몇 가지 방법으로 선발할 수 있나요?
주어진 정보를 바탕으로, 남학생 6명과 여학생 8명이 있는 수학 클럽이 있습니다. 주 수학 경시대회에 파견할 팀을 선발해야 하며, 팀은 6명의 학생으로 구성되어야 합니다. 제한 없이 팀을 몇 가지 방법으로 선발할 수 있는지 알아내야 합니다.

이 문제를 해결하기 위해 조합 공식을 사용할 수 있습니다. 조합 공식은 특정 순서 없이 몇 가지 방법으로 뽑을 수 있는지를 계산하는 데 사용됩니다. 이 경우, 우리는 남학생 6명과 여학생 8명 중에서 6명의 학생을 뽑아야 합니다.

조합 공식은 다음과 같습니다:

C(n, k) = n! / (k!(n-k)!)

여기서 C(n, k)는 n개 항목 중에서 k개를 선택하는 조합의 수를 나타내고, n!은 n의 팩토리얼을 나타내며, k
BLEU Score: 1.0265929471128496e-231



Generating response:  54%|█████▍    | 139/256 [18:15<15:22,  7.88s/step]   


Input: 자음이 하나 이상인 4글자 단어는 $A$, $B$, $C$, $D$, $E$로 몇 개나 만들 수 있나요? ($B$, $C$, $D$는 자음이며, 영어 단어뿐만 아니라 모든 단어가 유효하며, 글자를 두 번 이상 사용할 수 있습니다.)
Reference: 먼저 단어에 제한을 두지 않고 4글자로 된 모든 단어의 개수를 세어봅니다. 그런 다음 자음이 없는 4글자 단어의 개수를 세어봅니다. 그런 다음 뺄셈을 통해 답을 얻습니다.

단어의 각 글자는 $A$, $B$, $C$, $D$ 또는 $E$ 중 하나여야 하므로 단어에 제한이 없는 4글자 단어의 수는 $5\배수 5\배수 5\배수 5=625$입니다.  자음이 없는 단어의 각 글자는 $A$ 또는 $E$ 중 하나여야 합니다. 따라서 자음이 없는 모든 4글자 단어의 수는 $2\배수 2\배수 2\배수 2=16$입니다.  따라서 자음이 하나 이상 있는 4글자 단어의 수는 $625-16=609$입니다.
Generated: 자음이 하나 이상인 4글자 단어는 $A$, $B$, $C$, $D$, $E$로 몇 개나 만들 수 있나요? ($B$, $C$, $D$는 자음이며, 영어 단어뿐만 아니라 모든 단어가 유효하며, 글자를 두 번 이상 사용할 수 있습니다.)
자음이 하나 이상인 4글자 단어는 $A$, $B$, $C$, $D$, $E$로 몇 개나 만들 수 있나요? ($B$, $C$, $D$는 자음이며, 영어 단어뿐만 아니라 모든 단어가 유효하며, 글자를 두 번 이상 사용할 수 있습니다.)
BLEU Score: 1.0884584806948786e-231



Generating response: 100%|██████████| 256/256 [1:14:50<00:00, 17.54s/step]    


Input: 멜린다는 표준 6면 주사위 두 개를 굴려서 굴린 두 개의 숫자로 두 자리 숫자를 만듭니다. 예를 들어, 6과 3을 굴리면 36 또는 63을 만들 수 있습니다. 그녀가 10에서 20 사이의 정수를 만들 수 있을 확률은 얼마인가요? 답을 공통 분수로 표현하세요.
Reference: 주사위 중 하나 이상이 1에 나올 때만 가능합니다. 두 주사위가 모두 1이 아닐 확률은 $\left(\frac{5}{6}\right) \left(\frac{5}{6}\right) = \frac{25}{36}$입니다. 따라서 적어도 하나의 주사위가 1일 확률은 $1-\frac{25}{36} = \frac{11}{36}$입니다.
Generated: 멜린다는 표준 6면 주사위 두 개를 굴려서 굴린 두 개의 숫자로 두 자리 숫자를 만듭니다. 예를 들어, 6과 3을 굴리면 36 또는 63을 만들 수 있습니다. 그녀가 10에서 20 사이의 정수를 만들 수 있을 확률은 얼마인가요? 답을 공통 분수로 표현하세요.
10에서 20 사이의 정수를 만들기 위해서는 굴린 두 숫자의 합이 10에서 20 사이여야 합니다. 굴린 두 숫자의 합이 10에서 20 사이일 확률은 $\frac{11}{36}$입니다. 왜냐하면 굴린 두 숫자의 합이 10에서 20 사이일 확률이 $\frac{11}{36}$이기 때문입니다. 따라서 멜린다가 10에서 20 사이의 정수를 만들 확률은 $\frac{11}{36}$입니다.

10에서 20 사이의 정수를 만들기 위해서는 굴린 두 숫자의 합이 10에서 20 사이여야 합니다. 굴린 두 숫자의 합이 10에서 20 사이일 확률은 $\frac
BLEU Score: 9.766506671644074e-232



Generating response:   0%|          | 0/256 [00:00<?, ?step/s]Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
